# Run the Initial ANN hyperparameter search on Colab GPU

This notebook only exists to run `scripts/train_ann.py` on a GPU. All the actual logic
(model, data, metrics) lives in `src/marco/` and `scripts/` in the repo - this just clones
it, rebuilds the data it needs, and runs it.

Before running: **Runtime > Change runtime type > GPU** (T4 is fine).

You'll need a GitHub personal access token with `repo` scope (the repo is private),
stored as a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar) - not pasted
into a cell, so it doesn't end up saved in the notebook output.

In [ ]:
!nvidia-smi

## Clone the repo

In [ ]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")
repo_url = f"https://{token}@github.com/chamikaKity/Automated-Triage-of-Protein-Crystallisation-Images.git"

!git clone {repo_url} marco_project
%cd marco_project

## Install dependencies

Colab's torch/torchvision are already built against its CUDA driver. `pip install -e .`
with its normal dependency resolution would pull a PyPI torch wheel instead and likely
break GPU support, so we install the package with `--no-deps` and only add the one thing
Colab doesn't already have (`tfrecord`, needed to re-extract the dataset below).

In [ ]:
!pip install -e . --no-deps -q
!pip install tfrecord -q

## Rebuild the dataset

`data/raw/` isn't committed to git (it's large), but `scripts/download_extract.py` picks
its 15 shards deterministically (evenly spaced across the fixed 407-shard range), so
running it again here reproduces exactly the images `data/splits.csv` was built from -
no need to upload anything via Drive.

In [ ]:
!python scripts/download_extract.py

## Sanity check before spending GPU time

Confirm every path in `data/splits.csv` actually exists on disk.

In [ ]:
from pathlib import Path

import pandas as pd

splits = pd.read_csv("data/splits.csv")
missing = [p for p in splits["path"] if not Path(p).exists()]
print(f"{len(splits)} rows in splits.csv, {len(missing)} missing files")

## Run the hyperparameter search

Defaults: 12 configs x 12 epochs, over learning rate, batch size and backbone depth
(resnet18 vs resnet34). Every config is appended to `experiments/log.csv`; the best config
by recall@P_MIN is saved to `experiments/models/ann_initial.pt`.

In [ ]:
!python scripts/train_ann.py

## Get the results back to your laptop

`experiments/models/` is gitignored (model weights don't belong in git), so download these
two files by hand from the Colab file browser (left sidebar) into your local
`experiments/models/`:
- `experiments/models/ann_initial.pt`
- `experiments/models/ann_initial_history.json`

`experiments/log.csv` IS tracked, so push the new rows straight from here instead.

In [ ]:
!git config user.email "wwijesing1@gmail.com"
!git config user.name "Chamika Subashinie"
!git add experiments/log.csv
!git commit -m "Add Initial ANN hyperparameter search results (Colab GPU run)"
!git push